In [4]:
#Note: I've tried running some of this with modin to speed up bits of code, but am getting lots of errors. So going back to pandas.

import pandas as pd
#import modin.pandas as pd

## Loading and Cleaning

We'll start by loading the full data set, restricting to our training set (years 2014-2021), and converting fips_code into an integer and neighbors into a list of (integer) fips_code values. This will support some merging/functions below.

In [5]:
#Use modin to load the ../Data/Merged_Data/eaglei_noaa_era5.parquet file
df = pd.read_parquet('../Data/Merged_Data/eaglei_noaa_era5.parquet')

In [ ]:
#Drop rows corresponding to years 2022 and newer
#Skipping this for now until after features have been engineered
#df = df[df['YEAR'] < 2022]

In [6]:
#Convert fips_code to a float, round it to the nearest integer, and then convert to an int64
df['fips_code'] = df['fips_code'].astype(float).round().astype('int64')

In [7]:
#Use json to convert each value of neighbors into a list of integers
import json
df['neighbors'] = df['neighbors'].apply(lambda x: json.loads(x))

# NaN Values

Note that the following fips_code values are missing data for percent buried lines:
- 22101: St. Mary Parish, LA; it is adjacent to:
    - Iberia Parish (22045); this has a value of 0.1327997527742176
    - St. Martin Parish (22099); this has a value of 0.1628546644133763
    - Assumption Parish (22007); this has a value of 0.1593958603974588
    - Terrebonne Parish (22109); this has a value of 0.1939858072512728
    - If we use the average of the adjacent counties, it should have a value of 0.162259021. (But see the note below)
- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113, and should have a value of 0.1205137891106878
- 51720: Norton City, VA; this is in Wise County (fips_code 51195), and should have a value of 0.099491631

*Note that cach of these missing values should have been taken care of when running the county-level data scripts. Each case is somewhat unique and probably requires its own treatment. In general, it seems problematic to impute these values, so we're not going to write any code that could be used to try to impute them generally.*

The following fips_code values are missing data for Subregion (we can fill these in manually):
- 25001: Barnstable, MA; this has ZIP code 02630 and, according to epa.gov/egrid/power-profiler, it is in the NEWE subregion
- 51131: Northampton, VA; this has 16 ZIP codes, including 23307 which, according to the power-profiler, is in the RFCE subregion

The following fips_code values are missing data for Power_Dependent_Devices_DME_Mean
This should have originally been adjusted in the script that combined the emPower data, and this value propagated through the stages of data cleaning
 For now, we can look this value up from its original fips code:
- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113

The following fips_code values are missing ERA5 data. With the exception of Hudson, these are all adjacent to the ocean or a large body of water. 
It seems likely that the (rounded) centroid coordinates are not in the ERA5-Land dataset.
We could try to download more comprehensive ERA5 data to look up their values. 
Alternatively, we could drop these counties from the data.
Or, alternatively (again), we could impute values based on the average of their neighbors
- 12087: Monroe, FL; centroid is (-81.1 25.3) and neighbors are [12021, 12086]
- 25019: Nantucket, MA; centroid is (-70.1 41.3) but there are no neighbors
- 34017: Hudson, NJ; centroid is (-74.1 40.7) and neighbors are [36061, 34003, 34013]
- 37031: Carteret, NC; centroid is (-76.7 34.8) and neighbors are [37133, 37103, 37049]
- 48007: Aransas, TX; centroid is (-97 28.1) and neighbors are [48355, 48057, 48409, 48391]
- 51115: Mathews, VA; centroid is (-76.3 37.4) and neighbors are [51119, 51073]
- 51810: Virginia Beach, VA; centroid is (-76 36.7) and neighbors are [37053, 51710, 51550]
- 53029: Island, WA; centroid is (-122.5 48.2) and neighbors are [53061]

### Filling missing emPOWER data

We'll (re-)look up the emPOWER data for fips 46102 (which is listed as 46113 in the emPOWER data).

The (commented-out) code below was used to generate the following values. We'll fill these in manually
- 2023: 87.200000
- 2022: 79.083333
- 2021: 81.833333
- 2020: 84.666667
- 2019: 90.333333
- 2018: 92.666667
- 2017: 94.250000
- 2016: 98.750000
- 2015: 98.750000 (note that 2016 is the oldest data we have, so we'll assume the same values for 2015 and 2014)
- 2014: 98.750000

In [ ]:
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2023), 'Power_Dependent_Devices_DME_Mean'] = 87.200000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2022), 'Power_Dependent_Devices_DME_Mean'] = 79.083333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2021), 'Power_Dependent_Devices_DME_Mean'] = 81.833333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2020), 'Power_Dependent_Devices_DME_Mean'] = 84.666667
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2019), 'Power_Dependent_Devices_DME_Mean'] = 90.333333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2018), 'Power_Dependent_Devices_DME_Mean'] = 92.666667
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2017), 'Power_Dependent_Devices_DME_Mean'] = 94.250000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2016), 'Power_Dependent_Devices_DME_Mean'] = 98.750000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2015), 'Power_Dependent_Devices_DME_Mean'] = 98.750000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2014), 'Power_Dependent_Devices_DME_Mean'] = 98.750000


In [ ]:
## Note that the code below fills in missing emPower data for a specific fips value. This is an issue that should have been addressed in the previous cleaning scripts.
## After running the code, I found the specific missing values (above)
## No need to run this every time - I think it's fine to just replace the values manually

##Load the datasets
#empower2023 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2023_HHSemPOWERMapHistoricalDataset.csv')
#empower2022 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2022_HHSemPOWERMapHistoricalDataset.csv')
#empower2021 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2021_HHSemPOWERMapHistoricalDataset.csv')
#empower2020 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2020_HHSemPOWERMapHistoricalDataset.csv')
#empower2019 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2019_HHSemPOWERMapHistoricalDataset.csv')
#empower2018 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2018_HHSemPOWERMapHistoricalDataset.csv')
#empower2017 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2017_HHSemPOWERMapHistoricalDataset.csv')
#empower2016 = pd.read_csv('../Data/County_level_Variables/Empower_power_dependent_medical_devices/Empower_csvs/2016_HHSemPOWERMapHistoricalDataset.csv')

#empower_files = [empower2023, empower2022, empower2021, empower2020, empower2019, empower2018, empower2017, empower2016]

##Create an empty dataframe
#empower = pd.DataFrame()
#YEAR=2023

#for tdf in empower_files:
#    #Create a list of features we won't need (including all variables with "Medicare_Benes" in their name). Then drop these columns
#    nonfeatures = ['County_FIPS_Code', 'County','State_FIPS_Code', 'State']
#    nonfeatures.extend([col for col in tdf.columns if 'Medicare_Benes' in col])
#    tdf.drop(columns=[col for col in tdf.columns if col in nonfeatures], inplace=True)

#    #Change FIPS_Code of 46113 to 46102
#    tdf.loc[tdf['FIPS_Code'] == 46113, 'FIPS_Code'] = 46102

#    #Keep only the rows where FIPS_Code is equal to 46102
#    tdf = tdf[tdf['FIPS_Code'] == 46102]

#    #For each column in df that is an object, remove commas
#    for col in tdf.columns:
#        if tdf[col].dtype == 'object':
#            tdf[col] = tdf[col].str.replace(',', '')

#    #For each column in df, convert to an integer
#    for col in tdf.columns:
#        tdf[col] = pd.to_numeric(tdf[col], errors='coerce').fillna(0).astype(int)
    
#    #Append a new column computed by taking the mean of all columns with "Power_Dependent_Devices_DME" in their name
#    tdf['Power_Dependent_Devices_DME_Mean'] = tdf[[col for col in tdf.columns if 'Power_Dependent_Devices_DME' in col]].mean(axis=1)

#    #Drop all variables except for FIPS_Code and Power_Dependent_Devices_DME_Mean
#    tdf.drop(columns=[col for col in tdf.columns if col not in ['FIPS_Code', 'Power_Dependent_Devices_DME_Mean']], inplace=True)

#    #Add the YEAR as a variable
#    tdf['YEAR'] = YEAR
#    YEAR -= 1

#    #Concatenate df with empower
#    empower = pd.concat([empower, tdf])

##Reset the index of empower
#empower.reset_index(drop=True, inplace=True)

##Duplicate the last two rows of empower
#empower = pd.concat([empower, empower.tail(1)])
#empower = pd.concat([empower, empower.tail(1)])

##Reset the index of empower
#empower.reset_index(drop=True, inplace=True)

##In the last row of empower, change the value of YEAR to 2015
#empower.loc[empower.index[-2], 'YEAR'] = 2015
#empower.loc[empower.index[-1], 'YEAR'] = 2014

##Rename FIPS_Code as fips_code
#empower.rename(columns={'FIPS_Code': 'fips_code'}, inplace=True)

##In df look up NaN values in the Power_Dependent_Devices_DME_Mean from the empower dataframe, merging on fipcs_code and YEAR
#df = df.merge(empower, on=['fips_code', 'YEAR'], how='left')

## Combine the Power_Dependent_Devices_DME_Mean_x  and Power_Dependent_Devices_DME_Mean_y variables
#df['Power_Dependent_Devices_DME_Mean'] = df['Power_Dependent_Devices_DME_Mean_x'].fillna(df['Power_Dependent_Devices_DME_Mean_y'])

##Drop the Power_Dependent_Devices_DME_Mean_x and Power_Dependent_Devices_DME_Mean_y variables
#df.drop(columns=['Power_Dependent_Devices_DME_Mean_x', 'Power_Dependent_Devices_DME_Mean_y'], inplace=True)

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

### Filling missing buried power line data

- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113, and should have a value of 0.1205137891106878
- 51720: Norton City, VA; this is in Wise County (fips_code 51195), and should have a value of 0.099491631

In [13]:
df.loc[df['fips_code'] == 46102, 'Pct_Buried_Lines'] = 0.1205137891106878
df.loc[df['fips_code'] == 51720, 'Pct_Buried_Lines'] = 0.099491631
df.loc[df['fips_code'] == 22101, 'Pct_Buried_Lines'] = 0.162259021

### Filling missing Subregion data

- 25001: Barnstable, MA is in the NEWE subregion
- 51131: Northampton, VA is in the RFCE subregion

In [14]:
df.loc[df['fips_code'] == 25001, 'Subregion'] = 'NEWE'
df.loc[df['fips_code'] == 51131, 'Subregion'] = 'RFCE'

### Imputing missing ERA5 data

Several ocean-adjacent counties are missing ERA5 data:
- 12087: Monroe, FL; centroid is (-81.1 25.3) and neighbors are [12021, 12086]
- 25019: Nantucket, MA; centroid is (-70.1 41.3) but there are no neighbors
- 34017: Hudson, NJ; centroid is (-74.1 40.7) and neighbors are [36061, 34003, 34013]
- 37031: Carteret, NC; centroid is (-76.7 34.8) and neighbors are [37133, 37103, 37049]
- 48007: Aransas, TX; centroid is (-97 28.1) and neighbors are [48355, 48057, 48409, 48391]
- 51115: Mathews, VA; centroid is (-76.3 37.4) and neighbors are [51119, 51073]
- 51810: Virginia Beach, VA; centroid is (-76 36.7) and neighbors are [37053, 51710, 51550]
- 53029: Island, WA; centroid is (-122.5 48.2) and neighbors are [53061]

We'll impute these using the mean of the values from their neighboring counties (except for Nantucket... which we should probably just drop)

However, these means will be computed for all counties as part of feature engineering.

## Feature Engineering

Some features have already been engineered as part of the process of merging the eaglei, NOAA, and county-level predictors

Below, we'll engineer features on the merged dataset. For each fips code, we'll compute
- The mean value of each ERA5 predictor from all the neighboring counties
- The max value of each ERA5 predictor from all neighboring counties
- The total 12-hour and 24-hour precipitation and snowfall within each county

In [15]:
#Get the index of the first row in df
idx = df.index[0]

df.loc[idx]['datetime']

Timestamp('2019-01-01 18:00:00')

In [12]:
df.head(10)

,customers_out,datetime,YEAR,NAME,STUSPS,Pct_Buried_Lines,neighbors,Subregion,centroid_longitude,centroid_latitude,...,event_count Other,t2m,u10,v10,sf,tp,fips_code,Power_Dependent_Devices_DME_Mean,t2m_neighbors_mean,t2m_neighbors_max
0,0.708333,2019-01-01 18:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,285.815796,5.208618,-3.750549,0.0,6.510049e-03,10001,1609.583333,285.382202,286.556030
1,0.000000,2019-01-02 00:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,281.788086,2.149817,-2.922974,0.0,6.510049e-03,10001,1609.583333,281.696289,282.858398
2,0.000000,2019-01-02 06:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,278.574890,1.592340,-1.435226,0.0,0.000000e+00,10001,1609.583333,278.585632,279.362000
3,0.000000,2019-01-02 12:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,277.697937,-0.187393,-1.661896,0.0,8.523463e-07,10001,1609.583333,277.638031,278.381531
4,0.000000,2019-01-02 18:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,280.733704,-2.189249,0.626389,0.0,8.523463e-07,10001,1609.583333,280.707672,281.206360
5,0.000000,2019-01-03 00:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,278.930176,-2.953727,1.729481,0.0,1.314282e-06,10001,1609.583333,279.255035,279.736816
6,0.000000,2019-01-03 06:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,279.649231,0.254336,2.673775,0.0,3.635883e-05,10001,1609.583333,279.681793,280.473450
7,0.000000,2019-01-03 12:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,280.007416,3.485015,-0.212955,0.0,6.689727e-04,10001,1609.583333,279.929932,280.708588
8,0.416667,2019-01-03 18:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,280.605347,3.758061,-3.374830,0.0,6.689727e-04,10001,1609.583333,280.327362,281.251831
9,0.000000,2019-01-04 00:00:00,2019,Kent,DE,0.557942,"[10003, 10005, 24035, 24029, 24011]",RFCE,-75.6,39.1,...,0.0,278.266907,1.551849,-0.997677,0.0,6.689727e-04,10001,1609.583333,278.341766,279.186829


In [ ]:
#Compute the mean and maximum values of the ERA5 values of the neighbors for each row in df
for i in df.index:
    timeval = df.loc[i]['datetime']
    neighbors_list = df.loc[i]['neighbors']
    t2m_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 't2m']
    u10_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'u10']
    v10_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'v10']
    sf_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'sf']
    tp_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'tp']
    df.loc[i, 't2m_neighbors_mean'] = t2m_values.mean()
    df.loc[i, 't2m_neighbors_max'] = t2m_values.max()
    df.loc[i, 'u10_neighbors_mean'] = u10_values.mean()
    df.loc[i, 'u10_neighbors_max'] = u10_values.max()
    df.loc[i, 'v10_neighbors_mean'] = v10_values.mean()
    df.loc[i, 'v10_neighbors_max'] = v10_values.max()
    df.loc[i, 'sf_neighbors_mean'] = sf_values.mean()
    df.loc[i, 'sf_neighbors_max'] = sf_values.max()
    df.loc[i, 'tp_neighbors_mean'] = tp_values.mean()
    df.loc[i, 'tp_neighbors_max'] = tp_values.max()

### Computing weather features

- 12- and 24-hour total precipitation and snowfall
- Wind speed (from u and v components)
- Duration of weather events (from NOAA? or from ERA5? or both?)

... Maybe I should have done this for adjacent counties, too? or will XGboost figure that out?

In [190]:
# Do the following for each row in df where t2m is NaN:
# Look at the fips values in neighbors
# Find the value of t2m for the fips values with the same datetime value
# Compute the average of the t2m values
# Use this average as the missing t2m value
for i in df[df['t2m'].isnull()].index:
    # Get the datetime value of i
    datetime_value = df.loc[i]['datetime']
    print(datetime_value)
    # Make a list of the fips_code values in the neighbors variable
    neighbors_list = df.loc[i]['neighbors']
    print(neighbors_list)
    #For each value in neighbors_list, look up the value of t2m at the corresponding datetime value
    t2m_values = []
    for fips in neighbors_list:
        print(fips)
        t2m = df[(df['fips_code'] == fips) & (df['datetime'] == datetime_value)]['t2m']
        if not t2m.empty:
            t2m_values.append(t2m.values[0])
    #Compute the mean of t2m_values
    t2m_mean = sum(t2m_values) / len(t2m_values)

    #Set the t2m value of i to be equal to t2m_mean
    df.loc[i, 't2m'] = t2m_mean

time        latitude  longitude
2019-01-01  25.3      -81.1       2019-01-01
Name: datetime, dtype: datetime64[ns]
time        latitude  longitude
2019-01-01  25.3      -81.1        [12021, 12086]
Name: neighbors, dtype: object
[12021, 12086]


/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_3751/909352084.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  datetime_value = df.loc[i]['datetime']
/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_3751/909352084.py:11: PerformanceWarning: indexing past lexsort depth may impact performance.
  neighbors_list = df.loc[i]['neighbors']


ValueError: ('Lengths must match to compare', (27351570,), (2,))

In [134]:
#Count the number of NaN in each column of df
df.isnull().sum()

customers_out                           0
datetime                                0
YEAR                                    0
NAME                                    0
STUSPS                                  0
Pct_Buried_Lines                        0
neighbors                               0
Subregion                               0
centroid_longitude                      0
centroid_latitude                       0
centroid_rounded                        0
POPULATION                              0
BUILDVALUE                              0
AGRIVALUE                               0
AREA                                    0
SOVI_SCORE                              0
event_count SnowIce                     0
event_count Flood                       0
event_count Storm                       0
event_count Hurricane                   0
event_count Heat                        0
event_count Fire                        0
event_count Wind                        0
event_count Ocean                 

In [7]:
#Create a new column 'wind_speed' which is the square root of the sum of the squares of the 'u' and 'v' columns
df['wind_speed'] = (df['u10']**2 + df['v10']**2)**0.5

#Drop the 'u' and 'v' columns
df = df.drop(columns=['u10', 'v10'])